# 3.0 모델링
2.0에서 만든 연간 정답 표로 다음 해 잘버팀을 예측한다.
- 모델 고르기: 결과연도 2023 학습 → 2024 검증
- 최종 시험: 2023~2024 학습 → 2025 (딱 한 번)
- 비교 지표: 처음에는 상위 20% 적중률(추천용)로 비교했고, 쓰임새를 '상권 진단'으로 바꾸면서 ROC-AUC(전체 순서)와 확률 확인 표로 바꿈 → 6-1에서 전체 다시 비교
- 평가 지표: ROC-AUC. 상권을 넣으면 점수를 보여 주는 진단 도구라서, 50% 기준 없이 전체 상권의 순서가 맞는지가 중요하기 때문 (확률이 맞는지는 확률 확인 표로 따로 봄)

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/cafe_target_annual_2022_2025.csv")
print(df.shape)

(3504, 30)


## 1. 입력 칸 정하기와 연도별 나누기 ✅
입력: 작년 변화율, 작년에 카페가 유동인구보다 덜 늘었나, 폐업률, 상권 유형 (2.0 EDA 결론의 후보)
- 빈칸이 있는 행을 빼서 결과연도별 2023: 863행, 2024: 874행, 2025: 865행 (2022는 작년 값이 없어 전부 빠짐)
- 학습 2023 / 검증 2024 / 최종 시험 2025 (시험 연도는 마지막에 한 번만 봄)
- 정답 칸(다음 해 매출, 변화율, 중앙값, 잘버팀)과 상권 코드·이름은 입력에서 뺌

In [2]:
# H1 예측 버전 칸 다시 만들기 (2.0에서는 탐색용 표에만 만들었음)
df["작년_카페가_덜_늘었나"] = (df["작년_카페_증가율"] < df["작년_유동인구_증가율"]).astype(int)
df.loc[df["작년_카페_증가율"].isna() | df["작년_유동인구_증가율"].isna(), "작년_카페가_덜_늘었나"] = np.nan

입력칸 = ["작년_변화율", "작년_카페가_덜_늘었나", "폐업률", "상권_구분_코드_명"]   # ★ 모델이 보는 칸
df["상권_구분_코드_명"] = df["상권_구분_코드_명"].replace({"관광특구": "발달상권"})   # 사례가 적은 관광특구는 비슷한 발달상권과 합침
쓸행 = df.dropna(subset=입력칸).copy()
print(쓸행.groupby("결과연도").size())

train = 쓸행[쓸행["결과연도"] == 2023]     # 학습
val = 쓸행[쓸행["결과연도"] == 2024]       # 검증 (모델 고르기)
test = 쓸행[쓸행["결과연도"] == 2025]      # ★ 최종 시험: 마지막까지 열어 보지 않음
print(train.shape, val.shape)
print("검증 연도 잘버팀 비율:", round(val["잘버팀"].mean(), 3))

결과연도
2023    863
2024    874
2025    865
dtype: int64
(863, 31) (874, 31)
검증 연도 잘버팀 비율: 0.503


## 2. 기준선 ✅
- 동전 던지기: 약 50% (검증 연도 실제 잘버팀 비율 50.3%)
- 작년 변화율 규칙(가운데보다 높으면 1): 정확도 55.0%
- 작년 변화율 상위 20%를 뽑으면 62.3%가 실제로 잘 버팀 → 모델은 이 62.3%를 넘어야 의미가 있음
- 주 점수는 상위 20% 적중률: 추천받은 상권이 실제로 괜찮은지가 목적에 더 맞음

In [3]:
# 상위 20% 적중률: 점수가 높은 순서로 정확히 20%(개수 고정)를 뽑아, 그중 실제 잘버팀 비율
def 상위20(점수, 정답):
    n = int(round(len(점수) * 0.2))                                         # 뽑을 개수 (874곳이면 175곳)
    상위 = 점수.sort_values(ascending=False, kind="mergesort").index[:n]     # ★ 높은 순서로 줄 세워 n개
    return 정답.loc[상위].mean()

# 기준선: 작년 변화율이 가운데보다 높으면 '잘버팀'이라고 찍기
규칙 = (val["작년_변화율"] > val["작년_변화율"].median()).astype(int)
print("작년 변화율 규칙 - 정확도:", round((규칙 == val["잘버팀"]).mean(), 3))
print("작년 변화율 규칙 - 상위 20% 적중률:", round(상위20(val["작년_변화율"], val["잘버팀"]), 3))

작년 변화율 규칙 - 정확도: 0.55
작년 변화율 규칙 - 상위 20% 적중률: 0.623


## 3. 로지스틱 회귀 ✅
입력 4개로 2023년을 학습하고 2024년에 검증. 관광특구는 사례가 적어 발달상권과 합침.
- 4개 칸 모델: 정확도 54.3%, 상위 20% 적중률 60.0% → 기준선(작년 변화율 규칙 62.3%)보다 낮음
- 가중치: 작년 변화율 0.27, 폐업률 0.21, 카페가 덜 늘었나 0.16 (모두 +), 발달상권 +, 골목·전통시장 −
- 칸을 하나씩 더하면 상위 20%: 62.3% → 58.3% → 58.9% → 60.0%
- 상위 20%는 약 175곳이라 ±4%p는 우연으로도 생김 → 칸을 더해서 나아졌다고 볼 수 없음
- 관광특구를 합치기 전(59.4%)과 차이 없음

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline

숫자칸 = ["작년_변화율", "작년_카페가_덜_늘었나", "폐업률"]
글자칸 = ["상권_구분_코드_명"]

# 모델 만드는 틀: 숫자는 크기 맞추기, 글자는 0/1 칸으로 바꾸기 → 로지스틱 회귀
def 만들기(숫자, 글자):
    준비 = ColumnTransformer(
        [("숫자", StandardScaler(), 숫자)]
        + ([("글자", OneHotEncoder(handle_unknown="ignore"), 글자)] if 글자 else []))
    return make_pipeline(준비, LogisticRegression(max_iter=1000))

모델 = 만들기(숫자칸, 글자칸)
모델.fit(train[입력칸], train["잘버팀"])                                    # ★ 2023년으로 학습
확률 = pd.Series(모델.predict_proba(val[입력칸])[:, 1], index=val.index)     # 2024년 상권마다 잘버팀 확률
print("로지스틱 - 정확도:", round((모델.predict(val[입력칸]) == val["잘버팀"]).mean(), 3))
print("로지스틱 - 상위 20% 적중률:", round(상위20(확률, val["잘버팀"]), 3))   # ★ 62.3%와 비교

# 칸마다 배운 가중치 (+면 잘버팀 쪽, −면 반대 쪽)
print(pd.Series(모델[-1].coef_[0], index=모델[0].get_feature_names_out()).round(3))

로지스틱 - 정확도: 0.543
로지스틱 - 상위 20% 적중률: 0.6
숫자__작년_변화율             0.267
숫자__작년_카페가_덜_늘었나       0.162
숫자__폐업률                0.212
글자__상권_구분_코드_명_골목상권   -0.084
글자__상권_구분_코드_명_발달상권    0.268
글자__상권_구분_코드_명_전통시장   -0.189
dtype: float64


In [5]:
# 본인이 정한 순서대로 칸을 하나씩 더하며 비교
조합 = [
    (["작년_변화율"], []),
    (["작년_변화율", "작년_카페가_덜_늘었나"], []),
    (["작년_변화율", "작년_카페가_덜_늘었나", "폐업률"], []),
    (숫자칸, 글자칸),
]
for 숫자, 글자 in 조합:
    m = 만들기(숫자, 글자)
    m.fit(train[숫자 + 글자], train["잘버팀"])
    p = pd.Series(m.predict_proba(val[숫자 + 글자])[:, 1], index=val.index)
    print(숫자 + 글자, "정확도:", round((m.predict(val[숫자 + 글자]) == val["잘버팀"]).mean(), 3),
          "/ 상위 20%:", round(상위20(p, val["잘버팀"]), 3))

['작년_변화율'] 정확도: 0.535 / 상위 20%: 0.623
['작년_변화율', '작년_카페가_덜_늘었나'] 정확도: 0.541 / 상위 20%: 0.583
['작년_변화율', '작년_카페가_덜_늘었나', '폐업률'] 정확도: 0.566 / 상위 20%: 0.589
['작년_변화율', '작년_카페가_덜_늘었나', '폐업률', '상권_구분_코드_명'] 정확도: 0.543 / 상위 20%: 0.6


## 4. 랜덤포레스트 ✅
같은 입력 4개로 2023 학습 → 2024 검증. 관광특구는 사례가 18개라 가중치를 크게 외워서 발달상권과 합침 (둘 다 유동인구가 많은 대형 상권이라는 판단, 근거는 약함)
- 최소행 20: 상위 20% 65.1%, 개업률을 더하면 64.0% → 개업률은 도움이 안 됨
- 최소행만 바꿔도 57.7%(5) ~ 65.1%(20) ~ 62.3%(50)로 크게 흔들림
- 2024년 검증으로 고르면 최소행 20(65.1%)이지만, 고르는 데 쓴 점수라 최종 성능으로 보지 않음. 20과 50의 차이는 약 5곳
- 2023년 안의 교차검증(같은 해 다른 상권 기준)은 50이 64.0%로 높음 → 다음 해 검증을 대신하지 못해 보조 자료
- 결론: 로지스틱, 랜덤포레스트 모두 작년 변화율 규칙을 확실히 넘지 못함

In [6]:
from sklearn.ensemble import RandomForestClassifier

# 숲 만드는 틀: 숫자는 그대로, 글자는 0/1 칸으로 → 랜덤포레스트
def 숲(숫자, 글자, 최소행=20):
    준비 = ColumnTransformer(
        [("숫자", "passthrough", 숫자)]
        + ([("글자", OneHotEncoder(handle_unknown="ignore"), 글자)] if 글자 else []))
    return make_pipeline(준비, RandomForestClassifier(n_estimators=500, min_samples_leaf=최소행, random_state=42))

for 숫자 in [숫자칸, 숫자칸 + ["개업률"]]:          # 입력 4개, 그리고 개업률을 더한 5개
    m = 숲(숫자, 글자칸)
    m.fit(train[숫자 + 글자칸], train["잘버팀"])
    p = pd.Series(m.predict_proba(val[숫자 + 글자칸])[:, 1], index=val.index)
    print(숫자 + 글자칸, "정확도:", round((m.predict(val[숫자 + 글자칸]) == val["잘버팀"]).mean(), 3),
          "/ 상위 20%:", round(상위20(p, val["잘버팀"]), 3))

# 마지막 숲(5개 칸)에서 칸마다 얼마나 많이 쓰였나
print(pd.Series(m[-1].feature_importances_, index=m[0].get_feature_names_out()).round(3))

['작년_변화율', '작년_카페가_덜_늘었나', '폐업률', '상권_구분_코드_명'] 정확도: 0.559 / 상위 20%: 0.651
['작년_변화율', '작년_카페가_덜_늘었나', '폐업률', '개업률', '상권_구분_코드_명'] 정확도: 0.553 / 상위 20%: 0.64
숫자__작년_변화율             0.363
숫자__작년_카페가_덜_늘었나       0.059
숫자__폐업률                0.277
숫자__개업률                0.228
글자__상권_구분_코드_명_골목상권    0.025
글자__상권_구분_코드_명_발달상권    0.040
글자__상권_구분_코드_명_전통시장    0.007
dtype: float64


In [7]:
# 최소행 5 / 20 / 50으로 바꿔 보기 (고르기용이 아니라 흔들림 확인용)
for 최소행 in [5, 20, 50]:
    m = 숲(숫자칸, 글자칸, 최소행)
    m.fit(train[입력칸], train["잘버팀"])
    p = pd.Series(m.predict_proba(val[입력칸])[:, 1], index=val.index)
    print("최소행", 최소행, "정확도:", round((m.predict(val[입력칸]) == val["잘버팀"]).mean(), 3),
          "/ 상위 20%:", round(상위20(p, val["잘버팀"]), 3))

최소행 5 정확도: 0.543 / 상위 20%: 0.577
최소행 20 정확도: 0.559 / 상위 20%: 0.651
최소행 50 정확도: 0.556 / 상위 20%: 0.623


In [8]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

# 상위 20% 적중률을 교차검증 점수로 쓰기 위한 함수
def 상위20점수(모델, X, y):
    p = pd.Series(모델.predict_proba(X)[:, 1], index=X.index)
    return 상위20(p, y)

묶음 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)    # 2023년을 5묶음으로
for 최소행 in [5, 10, 20, 50, 100]:
    점수 = cross_val_score(숲(숫자칸, 글자칸, 최소행), train[입력칸], train["잘버팀"], cv=묶음, scoring=상위20점수)
    print("최소행", 최소행, "교차검증 상위 20% 평균:", round(점수.mean(), 3), "/ 흔들림:", round(점수.std(), 3))   # ★ 2023년 안에서만 비교

최소행 5 교차검증 상위 20% 평균: 0.613 / 흔들림: 0.046
최소행 10 교차검증 상위 20% 평균: 0.613 / 흔들림: 0.016
최소행 20 교차검증 상위 20% 평균: 0.636 / 흔들림: 0.037
최소행 50 교차검증 상위 20% 평균: 0.642 / 흔들림: 0.04
최소행 100 교차검증 상위 20% 평균: 0.63 / 흔들림: 0.037


## 5. 실험 B와 최종 방법 결정 규칙
- 입력 B: 작년 변화율, 작년 카페 증가율, 작년 유동인구 증가율, 작년 총매출 증가율, 점포당 연매출, 평균 카페 수, 직장인구, 폐업률, 상권 유형
- 점수 7개를 비교한다: 작년 변화율 규칙 1개 + 원래 입력 4개(A)로 만든 모델 3개(로지스틱, 랜덤포레스트 최소행 20, 50) + 확장 입력(B)으로 만든 같은 모델 3개
- 7개 모두 같은 상권들로 채점한다 (직장인구가 빈 상권을 빼서 2024년 872곳)
- 주 점수는 상위 20% 적중률
- 규칙보다 3%p(약 5곳) 이상 높고, 최소행 20과 50에서 모두 규칙보다 높을 때만 그 모델을 최종으로 고른다
- 아니면 가장 단순한 '작년 변화율 규칙'을 최종 방법으로 한다

**결과 ✅** (2024년 같은 872곳, 상위 20% 적중률)
- 규칙 62.1%
- 입력 A: 로지스틱 59.8%, 숲 20 63.8%, 숲 50 63.2%
- 입력 B: 로지스틱 57.5%, 숲 20 62.6%, 숲 50 62.1%
- 입력 A·B 중에는 규칙보다 3%p 이상 높은 모델이 없음 → 지금까지는 '작년 변화율 규칙'이 가장 나음 (최종 결정은 실험 C 뒤에)
- 입력 B는 정보를 더 넣었지만 오히려 낮아짐. B 로지스틱은 정확도는 가장 높지만(57.2%) 상위 20%는 가장 낮음 → 정확도만 보면 잘못 고를 수 있음

In [9]:
# 실험 B에 필요한 칸 만들기: 작년 총매출 증가율 (전년 총매출 = 전년 점포당 연매출 × 전년 평균 카페 수)
df["전년_연매출"] = df["전년_점포당_연매출"] * df["전년_평균_카페수"]
df["작년_총매출_증가율"] = (df["연매출"] - df["전년_연매출"]) / df["전년_연매출"]

B숫자칸 = ["작년_변화율", "작년_카페_증가율", "작년_유동인구_증가율", "작년_총매출_증가율",
          "점포당_연매출", "평균_카페수", "직장인구", "폐업률"]

# ★ 공정 비교: A와 B 모두 값이 있는 같은 행만 사용
B쓸행 = df.dropna(subset=list(set(숫자칸 + B숫자칸 + 글자칸))).copy()
trainB = B쓸행[B쓸행["결과연도"] == 2023]
valB = B쓸행[B쓸행["결과연도"] == 2024]
print(B쓸행.groupby("결과연도").size())

print("규칙 - 상위 20%:", round(상위20(valB["작년_변화율"], valB["잘버팀"]), 3))
for 이름, 숫자 in [("입력 A", 숫자칸), ("입력 B", B숫자칸)]:
    for 모델이름, m in [("로지스틱", 만들기(숫자, 글자칸)), ("숲 최소행20", 숲(숫자, 글자칸, 20)), ("숲 최소행50", 숲(숫자, 글자칸, 50))]:
        m.fit(trainB[숫자 + 글자칸], trainB["잘버팀"])
        p = pd.Series(m.predict_proba(valB[숫자 + 글자칸])[:, 1], index=valB.index)
        print(이름, 모델이름, "정확도:", round((m.predict(valB[숫자 + 글자칸]) == valB["잘버팀"]).mean(), 3),
              "/ 상위 20%:", round(상위20(p, valB["잘버팀"]), 3))

결과연도
2023    862
2024    872
2025    863
dtype: int64
규칙 - 상위 20%: 0.621
입력 A 로지스틱 정확도: 0.542 / 상위 20%: 0.598
입력 A 숲 최소행20 정확도: 0.563 / 상위 20%: 0.638
입력 A 숲 최소행50 정확도: 0.55 / 상위 20%: 0.632
입력 B 로지스틱 정확도: 0.572 / 상위 20%: 0.575
입력 B 숲 최소행20 정확도: 0.553 / 상위 20%: 0.626
입력 B 숲 최소행50 정확도: 0.556 / 상위 20%: 0.621


## 6. 실험 C: 최근 분기 흐름
- 새 칸: 4분기_전년동기_변화율(기준 연도 4분기 vs 전년 4분기), 분기_변화율_출렁임(분기별 전년 동기 변화율의 표준편차). 같은 분기끼리 비교라 계절 차이 없음
- 비교: 작년 변화율 규칙, 4분기 변화율 규칙, 입력 A 모델 3개, 입력 C(A + 새 칸 2개) 모델 3개. 모두 같은 상권으로 채점
- 규칙 두 개는 학습이 필요 없어서 결과연도 2023에서도 확인
- 이 실험으로 입력 실험은 끝낸다

**결과 ✅** (상위 20% 적중률)
- 규칙 비교: 2023년 작년 변화율 64.2% vs 4분기 67.1%, 2024년 62.3% vs 71.4% → 두 해 모두 4분기 규칙이 높음
- 입력 C 모델(2024): 로지스틱 69.1%, 숲 20 68.6%, 숲 50 69.1% → 입력 A(60.0~65.1%)보다 크게 높음
- 가능한 설명: 1년 동안 오른 상권은 연말 수준이 그해 평균보다 높고, 다음 해가 그 수준에서 출발함 (확인 안 됨)
- 4분기 값은 기준 연도 안의 값이라 미래 정보는 아님. 다만 실제 데이터는 분기 종료 몇 달 뒤에 공개됨

**최종 방법: 입력 C 로지스틱 회귀**
- 4분기 규칙과 2.3%p 차이(175곳 중 약 4곳)로 비슷함
- 쓰임새를 '상위 후보 추천'에서 '상권을 넣으면 점수와 위험도를 보여 주는 진단'으로 바꿈. 진단에는 상권마다 점수(확률)가 필요하고, 가중치로 이유도 설명할 수 있어서 모델을 고름 (기획안의 후보 상권 표·지도 대시보드와도 맞음)
- 이에 따라 주 점수를 ROC-AUC(전체 순서)와 확률 확인 표로, 상위 20% 적중률은 참고로 바꿈 → 6-1에서 전체 다시 비교
- 이 기준은 실험 C의 2024 결과를 본 뒤, 2025 결과를 보기 전에 바꿈. 2024 상위 20% 점수가 더 낮은 쪽을 고른 것
- 4분기 규칙은 2025 시험에서 비교 기준으로 같이 채점

In [10]:
# 분기 자료 다시 불러오기 (4분기 흐름을 만들려고)
q = pd.read_csv("../data/processed/cafe_full_2021_2025.csv",
                usecols=["기준_년분기_코드", "상권_코드", "당월_매출_금액", "유사_업종_점포_수"])
q["연도"] = q["기준_년분기_코드"] // 10
q["분기"] = q["기준_년분기_코드"] % 10
q["점포당_매출"] = q["당월_매출_금액"] / q["유사_업종_점포_수"]

# 같은 분기의 전년 값 붙이기 (연도 +1로 짝 맞춤) → 전년 동기 대비 변화율
전년 = q[["상권_코드", "연도", "분기", "점포당_매출"]].copy()
전년["연도"] = 전년["연도"] + 1
전년 = 전년.rename(columns={"점포당_매출": "전년동기_점포당_매출"})
q = pd.merge(q, 전년, on=["상권_코드", "연도", "분기"], how="left")
q["전년동기_변화율"] = (q["점포당_매출"] - q["전년동기_점포당_매출"]) / q["전년동기_점포당_매출"]

# 상권·연도마다: 분기 변화율의 출렁임 + 4분기 변화율
c = q.groupby(["상권_코드", "연도"]).agg(
    분기_변화율_출렁임=("전년동기_변화율", "std"),
    전년동기_분기수=("전년동기_변화율", "count")).reset_index()
c.loc[c["전년동기_분기수"] < 4, "분기_변화율_출렁임"] = np.nan          # 4개 분기가 다 있을 때만
q4c = q[q["분기"] == 4][["상권_코드", "연도", "전년동기_변화율"]].rename(columns={"전년동기_변화율": "4분기_전년동기_변화율"})
c = pd.merge(c, q4c, on=["상권_코드", "연도"], how="left")
dfC = pd.merge(df, c, on=["상권_코드", "연도"], how="left")             # ★ '연도' = 기준 연도라 미래 값 없음

C숫자칸 = 숫자칸 + ["4분기_전년동기_변화율", "분기_변화율_출렁임"]
C쓸행 = dfC.dropna(subset=C숫자칸 + 글자칸).copy()
trainC = C쓸행[C쓸행["결과연도"] == 2023]
valC = C쓸행[C쓸행["결과연도"] == 2024]
print(C쓸행.groupby("결과연도").size())

# 규칙 두 개: 결과연도 2023, 2024 모두에서
for 연도 in [2023, 2024]:
    v = C쓸행[C쓸행["결과연도"] == 연도]
    print(연도, "작년 변화율 규칙:", round(상위20(v["작년_변화율"], v["잘버팀"]), 3),
          "/ 4분기 규칙:", round(상위20(v["4분기_전년동기_변화율"], v["잘버팀"]), 3))

# 모델: 입력 A vs 입력 C (2023 학습 → 2024 검증)
for 이름, 숫자 in [("입력 A", 숫자칸), ("입력 C", C숫자칸)]:
    for 모델이름, m in [("로지스틱", 만들기(숫자, 글자칸)), ("숲 최소행20", 숲(숫자, 글자칸, 20)), ("숲 최소행50", 숲(숫자, 글자칸, 50))]:
        m.fit(trainC[숫자 + 글자칸], trainC["잘버팀"])
        p = pd.Series(m.predict_proba(valC[숫자 + 글자칸])[:, 1], index=valC.index)
        print(이름, 모델이름, "정확도:", round((m.predict(valC[숫자 + 글자칸]) == valC["잘버팀"]).mean(), 3),
              "/ 상위 20%:", round(상위20(p, valC["잘버팀"]), 3))

결과연도
2023    863
2024    874
2025    865
dtype: int64
2023 작년 변화율 규칙: 0.642 / 4분기 규칙: 0.671
2024 작년 변화율 규칙: 0.623 / 4분기 규칙: 0.714
입력 A 로지스틱 정확도: 0.543 / 상위 20%: 0.6
입력 A 숲 최소행20 정확도: 0.559 / 상위 20%: 0.651
입력 A 숲 최소행50 정확도: 0.556 / 상위 20%: 0.623
입력 C 로지스틱 정확도: 0.581 / 상위 20%: 0.691
입력 C 숲 최소행20 정확도: 0.585 / 상위 20%: 0.686
입력 C 숲 최소행50 정확도: 0.585 / 상위 20%: 0.691


## 6-1. 지표를 ROC-AUC로 바꿔 전체 다시 비교
쓰임새를 '상위 후보 추천'에서 '상권 진단'으로 바꿔서, 주 점수를 상위 20% 적중률에서 ROC-AUC(전체 순서)로 바꿈.
지금까지 비교한 방법 11개를 같은 상권(2023 학습 862곳, 2024 검증 872곳)으로 다시 채점해 결론이 유지되는지 확인한다.

In [11]:
from sklearn.metrics import roc_auc_score

# 모든 입력(A·B·C)에 값이 있는 같은 상권만
모든칸 = list(set(숫자칸 + B숫자칸 + C숫자칸 + 글자칸))
dfC["전년_연매출"] = dfC["전년_점포당_연매출"] * dfC["전년_평균_카페수"]
dfC["작년_총매출_증가율"] = (dfC["연매출"] - dfC["전년_연매출"]) / dfC["전년_연매출"]
공통 = dfC.dropna(subset=모든칸).copy()
tr = 공통[공통["결과연도"] == 2023]
va = 공통[공통["결과연도"] == 2024]
print("학습:", len(tr), "/ 검증:", len(va))

결과 = []
for 이름, 칸 in [("규칙: 작년 변화율", "작년_변화율"), ("규칙: 4분기 변화율", "4분기_전년동기_변화율")]:
    결과.append([이름, roc_auc_score(va["잘버팀"], va[칸]), 상위20(va[칸], va["잘버팀"])])

for 입력이름, 숫자 in [("입력 A", 숫자칸), ("입력 B", B숫자칸), ("입력 C", C숫자칸)]:
    for 모델이름, m in [("로지스틱", 만들기(숫자, 글자칸)), ("숲 최소행20", 숲(숫자, 글자칸, 20)), ("숲 최소행50", 숲(숫자, 글자칸, 50))]:
        m.fit(tr[숫자 + 글자칸], tr["잘버팀"])
        p = pd.Series(m.predict_proba(va[숫자 + 글자칸])[:, 1], index=va.index)
        결과.append([입력이름 + " " + 모델이름, roc_auc_score(va["잘버팀"], p), 상위20(p, va["잘버팀"])])

비교표 = pd.DataFrame(결과, columns=["방법", "ROC-AUC", "상위 20% 적중률"]).round(3)
print(비교표.sort_values("ROC-AUC", ascending=False).to_string(index=False))    # ★ AUC 높은 순

학습: 862 / 검증: 872
          방법  ROC-AUC  상위 20% 적중률
 규칙: 4분기 변화율    0.638       0.713
   입력 C 로지스틱    0.638       0.695
입력 C 숲 최소행50    0.630       0.690
입력 C 숲 최소행20    0.626       0.690
   입력 B 로지스틱    0.589       0.575
입력 A 숲 최소행50    0.587       0.632
   입력 A 로지스틱    0.581       0.598
입력 A 숲 최소행20    0.581       0.638
입력 B 숲 최소행50    0.580       0.621
입력 B 숲 최소행20    0.577       0.626
  규칙: 작년 변화율    0.560       0.621


## 6-2. 최종 모델의 확률이 믿을 만한가
모델이 '잘 버틸 확률 70%'라고 한 상권들이 실제로 약 70% 잘 버텼는지 2024년으로 확인한다.

In [12]:
from sklearn.metrics import brier_score_loss

검증모델 = 만들기(C숫자칸, 글자칸)
검증모델.fit(trainC[C숫자칸 + 글자칸], trainC["잘버팀"])
검증확률 = pd.Series(검증모델.predict_proba(valC[C숫자칸 + 글자칸])[:, 1], index=valC.index)

구간 = pd.cut(검증확률, [0, 0.3, 0.4, 0.5, 0.6, 0.7, 1])
표 = pd.DataFrame({"모델이 말한 확률(평균)": 검증확률.groupby(구간, observed=True).mean(),
                  "실제 잘버팀 비율": valC.groupby(구간, observed=True)["잘버팀"].mean(),
                  "상권 수": valC.groupby(구간, observed=True)["잘버팀"].size()}).round(3)
print(표)                                                                      # ★ 두 열이 비슷하면 확률을 믿을 만함
print("브라이어 점수:", round(brier_score_loss(valC["잘버팀"], 검증확률), 3), "(0에 가까울수록 좋음, 모두 50%라고 하면 0.25)")

            모델이 말한 확률(평균)  실제 잘버팀 비율  상권 수
(0.0, 0.3]          0.269      0.263    19
(0.3, 0.4]          0.364      0.276   123
(0.4, 0.5]          0.452      0.492   256
(0.5, 0.6]          0.546      0.490   249
(0.6, 0.7]          0.639      0.652   161
(0.7, 1.0]          0.803      0.727    66
브라이어 점수: 0.235 (0에 가까울수록 좋음, 모두 50%라고 하면 0.25)


## 6-3. 입력 칸끼리의 상관관계
가중치를 해석하기 전에, 입력 칸끼리 비슷한 정보를 담고 있는지 확인한다. 모델은 바꾸지 않고(이미 검증함), 상관이 높은 칸은 묶어서 해석한다.
**결과 ✅** (학습 연도 2023, 상관계수)
- 작년 변화율 ↔ 4분기 변화율 0.84 → 거의 같은 정보. 두 가중치는 따로 읽지 않고 '최근 흐름' 하나로 묶어서 해석
- 출렁임 ↔ 두 변화율 0.38~0.42 → 많이 오른 상권이 분기마다 더 들쭉날쭉한 편. 일부만 겹침
- 카페가 덜 늘었나, 폐업률 ↔ 나머지 0.00~0.14 → 흐름과 따로 노는 정보. 가중치를 각각 그대로 읽어도 됨
- 모델이 보는 정보는 사실상 세 가지: ① 최근 흐름 ② 출렁임 ③ 경쟁 상황(카페 증가, 폐업)

In [13]:
print(trainC[C숫자칸].corr().round(2))    # ★ 0.7보다 크면 두 칸이 비슷한 정보 → 가중치 해석 주의

              작년_변화율  작년_카페가_덜_늘었나   폐업률  4분기_전년동기_변화율  분기_변화율_출렁임
작년_변화율          1.00          0.05  0.05          0.84        0.42
작년_카페가_덜_늘었나    0.05          1.00  0.14          0.00       -0.07
폐업률             0.05          0.14  1.00          0.12        0.02
4분기_전년동기_변화율    0.84          0.00  0.12          1.00        0.38
분기_변화율_출렁임      0.42         -0.07  0.02          0.38        1.00


## 7. 최종 시험 (2025, 한 번만)
최종 모델(입력 C 로지스틱)은 6번에서 정함. 2023~2024 두 해로 다시 학습해 2025로 채점한다.
주 점수는 ROC-AUC와 확률 확인 표, 나머지는 참고용이며 결과를 보고 최종 모델을 바꾸지 않는다.

**결과 ✅** (2025, 865곳, 실제 잘버팀 49.8%)
- 최종 모델: ROC-AUC 0.667, 브라이어 0.232, 정확도 61.3%, 상위 20% 72.8% → 2024 검증(0.638)보다 조금 높아, 2024에 맞춰진 모델이 아님
- 3단계 등급: 위험 116곳 중 24.1%, 보통 565곳 중 47.6%, 양호 184곳 중 72.8%가 잘 버팀 → 양호가 위험보다 3배
- 확률 표도 대체로 맞음 (30% 이하 구간은 9곳뿐이라 튐)
- 참고: 4분기 규칙 ROC-AUC 0.679, 상위 20% 71.7% → 최종 모델과 사실상 비김(지표마다 이기는 쪽이 다름). 작년 변화율 규칙은 0.570, 57.2%로 약함
- 가중치: 4분기 변화율 +0.98, 작년 변화율 −0.35 → 상관 0.84라 묶어서 읽음: '1년 평균보다 연말까지 이어진 흐름'이 가장 큰 신호
- 폐업률 +0.14, 카페가 덜 늘었나 +0.13 → 경쟁이 줄어든 상권이 조금 더 잘 버팀. 출렁임 +0.02로 거의 영향 없음. 발달상권 +0.25, 골목·전통시장 약 −0.1

In [14]:
from sklearn.metrics import roc_auc_score, brier_score_loss

# 최종 모델: 입력 C 로지스틱을 2023~2024 두 해로 다시 학습
testC = C쓸행[C쓸행["결과연도"] == 2025]
train_all = C쓸행[C쓸행["결과연도"].isin([2023, 2024])]
최종모델 = 만들기(C숫자칸, 글자칸)
최종모델.fit(train_all[C숫자칸 + 글자칸], train_all["잘버팀"])
최종확률 = pd.Series(최종모델.predict_proba(testC[C숫자칸 + 글자칸])[:, 1], index=testC.index)

# ★ 주 점수
print("2025년 상권 수:", len(testC), "/ 실제 잘버팀 비율:", round(testC["잘버팀"].mean(), 3))
print("★ 최종 모델 - ROC-AUC:", round(roc_auc_score(testC["잘버팀"], 최종확률), 3),
      "/ 브라이어:", round(brier_score_loss(testC["잘버팀"], 최종확률), 3),
      "/ 정확도:", round((최종모델.predict(testC[C숫자칸 + 글자칸]) == testC["잘버팀"]).mean(), 3),
      "/ 상위 20%:", round(상위20(최종확률, testC["잘버팀"]), 3))

# ★ 확률이 2025년에도 맞는가
구간 = pd.cut(최종확률, [0, 0.3, 0.4, 0.5, 0.6, 0.7, 1])
print(pd.DataFrame({"모델이 말한 확률(평균)": 최종확률.groupby(구간, observed=True).mean(),
                    "실제 잘버팀 비율": testC.groupby(구간, observed=True)["잘버팀"].mean(),
                    "상권 수": testC.groupby(구간, observed=True)["잘버팀"].size()}).round(3))

# ★ 3단계 등급으로 묶어서 갈리는가
등급 = pd.cut(최종확률, [0, 0.4, 0.6, 1], labels=["위험", "보통", "양호"])
print(testC.groupby(등급, observed=True)["잘버팀"].agg(["mean", "count"]).round(3))

# 참고: 규칙 두 개
for 이름, 칸 in [("4분기 규칙", "4분기_전년동기_변화율"), ("작년 변화율 규칙", "작년_변화율")]:
    print("참고:", 이름, "- ROC-AUC:", round(roc_auc_score(testC["잘버팀"], testC[칸]), 3),
          "/ 상위 20%:", round(상위20(testC[칸], testC["잘버팀"]), 3))

# 최종 모델의 가중치 (+면 잘버팀 쪽) → 6-3의 세 묶음으로 나눠서 읽기
print(pd.Series(최종모델[-1].coef_[0], index=최종모델[0].get_feature_names_out()).round(3))

2025년 상권 수: 865 / 실제 잘버팀 비율: 0.498
★ 최종 모델 - ROC-AUC: 0.667 / 브라이어: 0.232 / 정확도: 0.613 / 상위 20%: 0.728
            모델이 말한 확률(평균)  실제 잘버팀 비율  상권 수
(0.0, 0.3]          0.245      0.333     9
(0.3, 0.4]          0.359      0.234   107
(0.4, 0.5]          0.453      0.429   275
(0.5, 0.6]          0.543      0.521   290
(0.6, 0.7]          0.639      0.719   121
(0.7, 1.0]          0.806      0.746    63
     mean  count
위험  0.241    116
보통  0.476    565
양호  0.728    184
참고: 4분기 규칙 - ROC-AUC: 0.679 / 상위 20%: 0.717
참고: 작년 변화율 규칙 - ROC-AUC: 0.57 / 상위 20%: 0.572
숫자__작년_변화율            -0.345
숫자__작년_카페가_덜_늘었나       0.130
숫자__폐업률                0.139
숫자__4분기_전년동기_변화율       0.980
숫자__분기_변화율_출렁임         0.023
글자__상권_구분_코드_명_골목상권   -0.097
글자__상권_구분_코드_명_발달상권    0.248
글자__상권_구분_코드_명_전통시장   -0.108
dtype: float64


## 8. 2026 전망 만들기 (대시보드용)
2025년 값을 입력으로 2026년 잘버팀을 예측한다. 시험이 끝났으므로 정답이 있는 모든 해(결과연도 2023~2025)로 다시 학습한다.
성능은 7번(2025 시험) 점수로 말하고, 결과는 위험·보통·양호 3단계와 '등급을 정한 이유'로 저장한다.

**결과 ✅**
- 2026 전망 대상 862곳: 보통 619 / 양호 129 / 위험 114
- 배포용 모델(결과연도 2023~2025로 학습) 가중치: 4분기 변화율 +1.17, 작년 변화율 −0.27(둘은 묶어서 읽음), 폐업률 +0.21, 카페가 덜 늘었나 +0.08, 출렁임 −0.05, 발달상권 +0.22
- 이 모델은 2025년 결과까지 배웠으므로 7번 점수(2023~2024로 학습한 모델)를 그대로 쓰지 않음 → 9번 2026 상반기 확인이 이 모델의 점수
- 저장: data/processed/cafe_forecast_2026.csv, models/final_logistic_c.joblib

In [15]:
import joblib, os
full = pd.read_csv("../data/processed/cafe_full_2021_2025.csv")
full["연도"] = full["기준_년분기_코드"] // 10
full["분기"] = full["기준_년분기_코드"] % 10

# (1) 상권·연도별 1년 요약 (2.0 셀 18과 같은 방식)
연 = full.groupby(["상권_코드", "연도"]).agg(
    상권_코드_명=("상권_코드_명", "first"), 상권_구분_코드_명=("상권_구분_코드_명", "first"),
    분기수=("분기", "nunique"), 연매출=("당월_매출_금액", "sum"), 평균_카페수=("유사_업종_점포_수", "mean"),
    폐업수=("폐업_점포_수", "sum"), 유동인구=("총_유동인구_수", "mean")).reset_index()
q4n = full[full["분기"] == 4][["상권_코드", "연도", "유사_업종_점포_수"]].rename(columns={"유사_업종_점포_수": "4분기_카페수"})
연 = pd.merge(연, q4n, on=["상권_코드", "연도"], how="left")
연 = 연[연["분기수"] == 4].copy()
연["점포당_연매출"] = 연["연매출"] / 연["평균_카페수"]

# (2) 기준 연도 2025 행 + 전년(2024) 값
기준 = 연[연["연도"] == 2025].copy()
전 = 연[연["연도"] == 2024][["상권_코드", "점포당_연매출", "평균_카페수", "유동인구"]]
전 = 전.rename(columns={"점포당_연매출": "전년_점포당_연매출", "평균_카페수": "전년_평균_카페수", "유동인구": "전년_유동인구"})
기준 = pd.merge(기준, 전, on="상권_코드", how="left")
기준["작년_변화율"] = (기준["점포당_연매출"] - 기준["전년_점포당_연매출"]) / 기준["전년_점포당_연매출"]
카페증가 = (기준["평균_카페수"] - 기준["전년_평균_카페수"]) / 기준["전년_평균_카페수"]
유동증가 = (기준["유동인구"] - 기준["전년_유동인구"]) / 기준["전년_유동인구"]
기준["작년_카페가_덜_늘었나"] = (카페증가 < 유동증가).astype(float)
기준.loc[카페증가.isna() | 유동증가.isna(), "작년_카페가_덜_늘었나"] = np.nan
기준["폐업률"] = 기준["폐업수"] / 기준["평균_카페수"]
기준["상권_구분_코드_명"] = 기준["상권_구분_코드_명"].replace({"관광특구": "발달상권"})
기준 = pd.merge(기준, c, on=["상권_코드", "연도"], how="left")        # 4분기 변화율·출렁임 (6번 실험 C 셀에서 만든 c)

# (3) 대상 조건은 지금까지와 같음
대상 = 기준[(기준["4분기_카페수"] >= 5) & (기준["점포당_연매출"] >= 4_000_000)].dropna(subset=C숫자칸 + 글자칸).copy()
print("2026 전망 대상 상권:", len(대상))

# (4) 배포용 모델: 정답이 있는 모든 해(결과연도 2023~2025)로 다시 학습
배포모델 = 만들기(C숫자칸, 글자칸)
배포모델.fit(C쓸행[C숫자칸 + 글자칸], C쓸행["잘버팀"])
대상["잘버팀_확률"] = 배포모델.predict_proba(대상[C숫자칸 + 글자칸])[:, 1]
대상["등급"] = pd.cut(대상["잘버팀_확률"], [0, 0.4, 0.6, 1], labels=["위험", "보통", "양호"]).astype(str)
print(대상["등급"].value_counts())

# (5) 등급을 정한 이유: 칸마다 점수에 더하고 뺀 양 (가중치 × 크기를 맞춘 값). 상관이 높은 두 흐름 칸은 묶음
변환 = 배포모델[0].transform(대상[C숫자칸 + 글자칸])
기여 = pd.DataFrame(변환 * 배포모델[-1].coef_[0], columns=배포모델[0].get_feature_names_out(), index=대상.index)
대상["이유_최근흐름"] = 기여["숫자__작년_변화율"] + 기여["숫자__4분기_전년동기_변화율"]
대상["이유_폐업"] = 기여["숫자__폐업률"]
대상["이유_카페증가"] = 기여["숫자__작년_카페가_덜_늘었나"]
대상["이유_출렁임"] = 기여["숫자__분기_변화율_출렁임"]
대상["이유_상권유형"] = 기여[[k for k in 기여.columns if k.startswith("글자__")]].sum(axis=1)

# (6) 저장: 대시보드가 읽을 표 + 모델
저장칸 = ["상권_코드", "상권_코드_명", "상권_구분_코드_명", "잘버팀_확률", "등급", "점포당_연매출", "평균_카페수", "4분기_카페수",
        "작년_변화율", "4분기_전년동기_변화율", "폐업률", "작년_카페가_덜_늘었나",
        "이유_최근흐름", "이유_폐업", "이유_카페증가", "이유_출렁임", "이유_상권유형"]
os.makedirs("../models", exist_ok=True)
대상[저장칸].to_csv("../data/processed/cafe_forecast_2026.csv", index=False, encoding="utf-8-sig")
joblib.dump(배포모델, "../models/final_logistic_c.joblib")
print(대상[저장칸].shape)
print(pd.Series(배포모델[-1].coef_[0], index=배포모델[0].get_feature_names_out()).round(3))

2026 전망 대상 상권: 862
등급
보통    619
양호    129
위험    114
Name: count, dtype: int64
(862, 17)
숫자__작년_변화율            -0.266
숫자__작년_카페가_덜_늘었나       0.075
숫자__폐업률                0.206
숫자__4분기_전년동기_변화율       1.168
숫자__분기_변화율_출렁임        -0.048
글자__상권_구분_코드_명_골목상권   -0.092
글자__상권_구분_코드_명_발달상권    0.217
글자__상권_구분_코드_명_전통시장   -0.086
dtype: float64
